In [122]:
import pandas as pd
from pprint import pprint
#filter to only the rows where the model is resnet50
# gb = cc.groupby(["backbone", "pooling", "sampler", "weight_config", "fulltune"])
coralcam_groups = ['aggression', 'biting']
fishfollow_groups = ['habitat', 'movement', 'bites', 'social_interaction', 'not_visible']
cc = pd.read_csv("../results/new_grouping/coralcam_results_label_tolerance_7.csv")
ff = pd.read_csv("../results/new_grouping/fishfollow_results_label_tolerance_7.csv")

In [123]:
ff['weight_config']

0                               weight_method='uniform'
1     {'weight_method': 'focal_loss', 'focal_loss_al...
2                          {'weight_method': 'uniform'}
3     {'weight_method': 'focal_loss', 'focal_loss_al...
4                          {'weight_method': 'uniform'}
5     weight_method='focal_loss' focal_loss_gamma=5....
6                               weight_method='uniform'
7     {'weight_method': 'focal_loss', 'focal_loss_al...
8                          {'weight_method': 'uniform'}
9     {'weight_method': 'focal_loss', 'focal_loss_al...
10    {'weight_method': 'focal_loss', 'focal_loss_al...
11                         {'weight_method': 'uniform'}
12                         {'weight_method': 'uniform'}
13                              weight_method='uniform'
14    weight_method='focal_loss' focal_loss_gamma=5....
15                         {'weight_method': 'uniform'}
16                         {'weight_method': 'uniform'}
17    {'weight_method': 'focal_loss', 'focal_los

In [124]:
import re
def parse_weight_config(weight_config):
    s = str(weight_config)

    # Case 1: plain string (ex: weight_method='focal_loss')
    match1 = re.search(
        r"weight_method\s*[:=]\s*['\"]?([A-Za-z_]+)",
        s
    )

    # Case 2: dict-like (ex: {'weight_method': 'focal_loss', ...})
    match2 = re.search(
        r"'weight_method'\s*:\s*['\"]?([A-Za-z_]+)",
        s
    )

    if match1:
        return match1.group(1)
    if match2:
        return match2.group(1)
    return None

cc['ci_strategy'] = cc['weight_config'].apply(parse_weight_config)
ff['ci_strategy'] = ff['weight_config'].apply(parse_weight_config)

cc.fillna({"freeze_backbone": False}, inplace=True)
cc['fulltune_status'] = cc['fulltune'] & ~cc['freeze_backbone']

ff.fillna({"freeze_backbone": False}, inplace=True)
ff['fulltune_status'] = ff['fulltune'] & ~ff['freeze_backbone']

/tmp/ipykernel_813006/195439653.py:26: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cc.fillna({"freeze_backbone": False}, inplace=True)
/tmp/ipykernel_813006/195439653.py:29: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ff.fillna({"freeze_backbone": False}, inplace=True)


In [91]:
ff.groupby(['backbone', 'ci_strategy']).groups

{('dinov3_large', 'focal_loss'): [1, 3], ('dinov3_large', 'uniform'): [0, 2], ('resnet50', 'focal_loss'): [9, 17], ('resnet50', 'uniform'): [8, 15, 16], ('resnet50', nan): [18], ('videomae', 'focal_loss'): [10, 14], ('videomae', 'uniform'): [11, 12, 13], ('vjepa2', 'focal_loss'): [5, 7], ('vjepa2', 'uniform'): [4, 6]}

In [132]:
def get_backbone_table(df: pd.DataFrame, group_names, max_metric):
    df = df[df['fulltune_status'] == False]
    best_rows = df.loc[df.groupby("backbone")[max_metric].idxmax()]
    return best_rows[["backbone"] + ["f1_macro", "mAP"] +
            [f"{group}_f1_macro" for group in group_names] +
            [f"{group}_mAP" for group in group_names]]
    

In [133]:
get_backbone_table(cc, coralcam_groups, "f1_macro")

,backbone,f1_macro,mAP,aggression_f1_macro,biting_f1_macro,aggression_mAP,biting_mAP
1,dinov3_large,0.329175,0.321502,0.006579,0.490473,0.001237,0.481634
9,resnet50,0.285857,0.158666,0.000000,0.428786,0.000642,0.237678
13,videomae,0.362011,0.130226,0.310870,0.387581,0.063816,0.163431
7,vjepa2,0.356212,0.252290,0.294118,0.387260,0.262049,0.247411


In [134]:
get_backbone_table(ff, fishfollow_groups, "f1_macro")

,backbone,f1_macro,mAP,habitat_f1_macro,movement_f1_macro,bites_f1_macro,social_interaction_f1_macro,not_visible_f1_macro,habitat_mAP,movement_mAP,bites_mAP,social_interaction_mAP,not_visible_mAP
1,dinov3_large,0.379678,0.307225,0.593821,0.458118,0.123250,0.036066,0.457943,0.534870,0.384305,0.003993,0.036394,0.419410
8,resnet50,0.360611,0.268328,0.571164,0.460216,0.081677,0.022860,0.405485,0.440841,0.353104,0.003712,0.029329,0.359760
14,videomae,0.370142,0.301287,0.552342,0.453869,0.104541,0.085402,0.486447,0.505345,0.382440,0.004217,0.051644,0.424197
7,vjepa2,0.375297,0.334886,0.566341,0.484327,0.118818,0.000000,0.401754,0.551765,0.444073,0.004929,0.031648,0.431417


In [135]:
def get_binary_pivot_table(df: pd.DataFrame, pivot, pivot_values: list, max_metric): 
    df = df[df['fulltune_status'] == False]
    best_rows = df.loc[df.groupby(["backbone", pivot])[max_metric].idxmax()]
    #pivot the table to have ci_strategy as columns and backbone as rows
    pivoted = best_rows.pivot(index="backbone", columns=pivot, values=["f1_macro", "mAP", 'precision_macro', 'recall_macro'])
    assert len(pivot_values) == 2, "Only supports two pivot values for now"
    pivoted["delta_precision_macro"] = (
        pivoted[("precision_macro", pivot_values[0])]
        - pivoted[("precision_macro", pivot_values[1])]
    )
    pivoted["delta_recall_macro"] = (
        pivoted[("recall_macro", pivot_values[0])]
        - pivoted[("recall_macro", pivot_values[1])]
    )
    pivoted["delta_f1_macro"] = (
        pivoted[("f1_macro", pivot_values[0])]
        - pivoted[("f1_macro", pivot_values[1])]
    )
    #delete precision_macro and recall_macro columns
    pivoted = pivoted.drop(
        columns=[("precision_macro", pivot_values[0]), ("precision_macro", pivot_values[1]), ("recall_macro", pivot_values[0]), ("recall_macro", pivot_values[1])])
    return pivoted.reset_index()


In [136]:
get_binary_pivot_table(cc, "ci_strategy", ["focal_loss", "uniform"], "f1_macro")

backbone   f1_macro                  mAP            \
ci_strategy               focal_loss   uniform focal_loss   uniform   
0            dinov3_large   0.329175  0.324976   0.321502  0.315239   
1                resnet50   0.285857  0.285276   0.158666  0.161307   
2                videomae   0.288407  0.362011   0.118630  0.130226   
3                  vjepa2   0.356212  0.352122   0.252290  0.234684   

            delta_precision_macro delta_recall_macro delta_f1_macro  
ci_strategy                                                          
0                       -0.000911           0.066890       0.004199  
1                        0.001197           0.010937       0.000581  
2                       -0.098642          -0.074850      -0.073604  
3                       -0.009901           0.061642       0.004090

In [137]:
get_binary_pivot_table(ff, "ci_strategy", ["focal_loss", "uniform"], "f1_macro")

backbone   f1_macro                  mAP            \
ci_strategy               focal_loss   uniform focal_loss   uniform   
0            dinov3_large   0.379678  0.373225   0.307225  0.313492   
1                resnet50   0.360420  0.360611   0.267906  0.268328   
2                videomae   0.370142  0.357933   0.301287  0.303900   
3                  vjepa2   0.375297  0.356103   0.334886  0.348479   

            delta_precision_macro delta_recall_macro delta_f1_macro  
ci_strategy                                                          
0                       -0.044236           0.129178       0.006453  
1                       -0.036799           0.166843      -0.000191  
2                       -0.056325           0.154537       0.012208  
3                       -0.093010           0.120854       0.019195

In [138]:
get_binary_pivot_table(cc, "pooling", ['attention', 'mean'], "f1_macro")

backbone  f1_macro                 mAP            \
pooling               attention      mean attention      mean   
0        dinov3_large  0.329175  0.237616  0.321502  0.108423   
1            resnet50  0.285857  0.249886  0.158666  0.107070   
2            videomae  0.362011  0.192891  0.130226  0.111989   
3              vjepa2  0.241273  0.356212  0.100342  0.252290   

        delta_precision_macro delta_recall_macro delta_f1_macro  
pooling                                                          
0                    0.059874           0.155977       0.091558  
1                    0.036827          -0.026562       0.035972  
2                    0.183486          -0.171300       0.169120  
3                   -0.079213          -0.113792      -0.114939

In [139]:
get_binary_pivot_table(ff, "pooling", ['attention', 'mean'], "f1_macro")

backbone  f1_macro                 mAP            \
pooling               attention      mean attention      mean   
0        dinov3_large  0.379678  0.364806  0.307225  0.299616   
1            resnet50  0.360611  0.349642  0.268328  0.264370   
2            videomae  0.370142  0.338825  0.301287  0.299193   
3              vjepa2  0.371680  0.375297  0.297917  0.334886   

        delta_precision_macro delta_recall_macro delta_f1_macro  
pooling                                                          
0                    0.005585           0.042149       0.014872  
1                    0.033261          -0.062978       0.010969  
2                    0.006143           0.111556       0.031317  
3                   -0.032490           0.079891      -0.003617

In [140]:
def get_resnet_table(df, group_names, max_metric):
    dino = df[df['backbone'] == 'dinov3_large']
    resnet_frozen = df[(df['backbone'] == 'resnet50') & (df['fulltune_status'] == False)]
    resnet_fulltune = df[(df['backbone'] == 'resnet50') & (df['fulltune_status'] == True)]

    dino_best = dino.loc[dino[max_metric].idxmax()]
    resnet_frozen_best = resnet_frozen.loc[resnet_frozen[max_metric].idxmax()]
    resnet_fulltune_best = resnet_fulltune.loc[resnet_fulltune[max_metric].idxmax()]
    
    
    # columns you care about
    cols = (
        ["backbone", "fulltune_status", "f1_macro", "mAP"]
        + [f"{g}_f1_macro" for g in group_names]
        + [f"{g}_mAP" for g in group_names]
    )

    # each *_best is a Series → turn into 1-row DataFrame with .to_frame().T
    table = pd.concat(
        [row[cols].to_frame().T for row in [dino_best, resnet_frozen_best, resnet_fulltune_best]],
        ignore_index=True,
    )

    return table

In [141]:
get_resnet_table(cc, coralcam_groups, "f1_macro")

,backbone,fulltune_status,f1_macro,mAP,aggression_f1_macro,biting_f1_macro,aggression_mAP,biting_mAP
0,dinov3_large,False,0.329175,0.321502,0.006579,0.490473,0.001237,0.481634
1,resnet50,False,0.285857,0.158666,0.0,0.428786,0.000642,0.237678
2,resnet50,True,0.357826,0.242816,0.0,0.53674,0.000642,0.363903


In [142]:
get_resnet_table(ff, fishfollow_groups, "f1_macro")

,backbone,fulltune_status,f1_macro,mAP,habitat_f1_macro,movement_f1_macro,bites_f1_macro,social_interaction_f1_macro,not_visible_f1_macro,habitat_mAP,movement_mAP,bites_mAP,social_interaction_mAP,not_visible_mAP
0,dinov3_large,False,0.379678,0.307225,0.593821,0.458118,0.12325,0.036066,0.457943,0.53487,0.384305,0.003993,0.036394,0.41941
1,resnet50,False,0.360611,0.268328,0.571164,0.460216,0.081677,0.02286,0.405485,0.440841,0.353104,0.003712,0.029329,0.35976
2,resnet50,True,0.321379,0.298217,0.549549,0.40456,0.040642,0.0,0.384554,0.527068,0.380348,0.003259,0.028926,0.355172
